In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Original dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecur

,Missing Values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [4]:
# Remove duplicate customer records
df = df.drop_duplicates(subset="customerID").copy()

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Fill missing values
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Remove customer identifier because it is not a useful model feature
df = df.drop(columns=["customerID"])

print("Data cleaning completed!")
print("Cleaned dataset shape:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

display(df.head())

Data cleaning completed!
Cleaned dataset shape: (7043, 20)
Missing values remaining: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0,"Yes": 1})
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (7043, 19)
Target shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [6]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
print("Categorical columns:")
print(categorical_columns)
print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [7]:
X_encoded = pd.get_dummies(X,columns=categorical_columns,drop_first=True,dtype=int)
print("Encoding completed!")
print("Original feature shape:", X.shape)
print("Encoded feature shape:", X_encoded.shape)
display(X_encoded.head())

Encoding completed!
Original feature shape: (7043, 19)
Encoded feature shape: (7043, 30)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=42,stratify=y)
print("Train-test split completed!")
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Train-test split completed!

X_train shape: (5634, 30)
X_test shape: (1409, 30)
y_train shape: (5634,)
y_test shape: (1409,)


In [10]:
# Get numerical feature columns
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
# Create scaler
scaler = StandardScaler()
# Fit only on training data
X_train = X_train.copy()
X_test = X_test.copy()

X_train.loc[:, numeric_features] = scaler.fit_transform(X_train[numeric_features])
# Transform test data using training parameters
X_test.loc[:, numeric_features] = scaler.transform(X_test[numeric_features])
print("Feature scaling completed!")
print("Training numerical feature means:")
display(X_train[numeric_features].mean().round(3))

print("\nTraining numerical feature standard deviations:")
display(X_train[numeric_features].std().round(3))

Feature scaling completed!
Training numerical feature means:


,0
SeniorCitizen,0.0
tenure,-0.0
MonthlyCharges,-0.0
TotalCharges,0.0
gender_Male,-0.0
Partner_Yes,0.0
Dependents_Yes,0.0
PhoneService_Yes,-0.0
MultipleLines_No phone service,0.0
MultipleLines_Yes,-0.0



Training numerical feature standard deviations:


,0
SeniorCitizen,1.0
tenure,1.0
MonthlyCharges,1.0
TotalCharges,1.0
gender_Male,1.0
Partner_Yes,1.0
Dependents_Yes,1.0
PhoneService_Yes,1.0
MultipleLines_No phone service,1.0
MultipleLines_Yes,1.0


In [12]:
feature_summary = pd.DataFrame({
    "Feature Type": [
        "Original numerical features",
        "Original categorical features",
        "Encoded numerical/dummy features",
        "Target variable"
    ],
    "Before Processing": [
        len(numerical_columns),
        len(categorical_columns),
        0,
        1
    ],
    "After Processing": [
        len(numeric_features),
        0,
        X_encoded.shape[1],
        1
    ]
})

display(feature_summary)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 40)

print("Original records:", len(df))
print("Encoded features:", X_encoded.shape[1])

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts())

print("\nTesting churn distribution:")
print(y_test.value_counts())

,Feature Type,Before Processing,After Processing
0,Original numerical features,4,30
1,Original categorical features,15,0
2,Encoded numerical/dummy features,0,30
3,Target variable,1,1


FINAL PREPROCESSING SUMMARY
Original records: 7043
Encoded features: 30

Training data: (5634, 30)
Testing data: (1409, 30)
Training labels: (5634,)
Testing labels: (1409,)

Training churn distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Testing churn distribution:
Churn
0    1035
1     374
Name: count, dtype: int64


In [14]:
# Scale only original numerical features, not dummy columns

numeric_features = numerical_columns

scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train.loc[:, numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test.loc[:, numeric_features] = scaler.transform(X_test[numeric_features])
print("Original numerical features scaled successfully!")
print(numeric_features)

# Convert target series to DataFrames
y_train_df = y_train.to_frame(name="Churn")
y_test_df = y_test.to_frame(name="Churn")

# Export all four CSV files
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train_df.to_csv("y_train.csv", index=False)
y_test_df.to_csv("y_test.csv", index=False)

print("All four CSV files exported successfully!")

print("\nGenerated files:")
print("1. X_train.csv")
print("2. X_test.csv")
print("3. y_train.csv")
print("4. y_test.csv")

Original numerical features scaled successfully!
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
All four CSV files exported successfully!

Generated files:
1. X_train.csv
2. X_test.csv
3. y_train.csv
4. y_test.csv


In [16]:
import os

files = [
    "X_train.csv",
    "X_test.csv",
    "y_train.csv",
    "y_test.csv"
]

for file in files:
    print(file, "->", os.path.getsize(file), "bytes")

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert X_train.shape[1] == X_test.shape[1]
assert list(X_train.columns) == list(X_test.columns)

assert not X_train.isnull().any().any()
assert not X_test.isnull().any().any()

assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

print("All preprocessing validation checks passed!")

X_train.csv -> 3301584 bytes
X_test.csv -> 826275 bytes
y_train.csv -> 11274 bytes
y_test.csv -> 2824 bytes
All preprocessing validation checks passed!
